## 1. Import Library

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Data Collection 

In [19]:
user_data = pd.read_csv("u.data",sep = "\t",names=['userid','itemid','rating','timestamp'])
user_data

,userid,itemid,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [17]:
Items_data = pd.read_csv("u.item",sep = "|",encoding='latin-1',header=None)
Items_data = Items_data[[0,1]]
Items_data.columns = ["itemid",'title']
Items_data

,itemid,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)
...,...,...
1677,1678,Mat' i syn (1997)
1678,1679,B. Monkey (1998)
1679,1680,Sliding Doors (1998)
1680,1681,You So Crazy (1994)


In [21]:
movie_data = pd.merge(user_data,Items_data,on = 'itemid')

In [23]:
movie_data.head()

,userid,itemid,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [26]:
movie_data['timestamp'] = pd.to_datetime(movie_data.timestamp,unit='s')

In [27]:
movie_data

,userid,itemid,rating,timestamp,title
0,196,242,3,1997-12-04 15:55:49,Kolya (1996)
1,186,302,3,1998-04-04 19:22:22,L.A. Confidential (1997)
2,22,377,1,1997-11-07 07:18:36,Heavyweights (1994)
3,244,51,2,1997-11-27 05:02:03,Legends of the Fall (1994)
4,166,346,1,1998-02-02 05:33:16,Jackie Brown (1997)
...,...,...,...,...,...
99995,880,476,3,1997-11-22 05:10:44,"First Wives Club, The (1996)"
99996,716,204,5,1997-11-17 19:39:03,Back to the Future (1985)
99997,276,1090,1,1997-09-20 22:49:55,Sliver (1993)
99998,13,225,2,1997-12-17 22:52:36,101 Dalmatians (1996)


In [29]:
movie_data.title.nunique()

1664

## Recommendation Enginr

### 1. Popularity Based

In [ ]:
movie_data.agg()

In [46]:
def Popularity_RE(top_n = 10):
    pop = movie_data.groupby(by = 'title').agg({'rating':['mean','count']})
    pop.columns = ['avg_rating','count']
    pop = pop[pop['count']> 50]
    pop = pop.sort_values(by = 'avg_rating',ascending = False)
    return pop.head(top_n).index.tolist()

Popularity_RE()

['Close Shave, A (1995)',
 "Schindler's List (1993)",
 'Wrong Trousers, The (1993)',
 'Casablanca (1942)',
 'Wallace & Gromit: The Best of Aardman Animation (1996)',
 'Shawshank Redemption, The (1994)',
 'Rear Window (1954)',
 'Usual Suspects, The (1995)',
 'Star Wars (1977)',
 '12 Angry Men (1957)']

## 2. Trend Based as per Time

In [56]:
movie_data.sort_values(by = 'timestamp',ascending = False).head()

,userid,itemid,rating,timestamp,title
52134,729,333,4,1998-04-22 23:10:38,"Game, The (1997)"
46574,729,328,3,1998-04-22 23:10:38,Conspiracy Theory (1997)
46773,729,689,4,1998-04-22 23:10:38,"Jackal, The (1997)"
79208,729,272,4,1998-04-22 23:10:38,Good Will Hunting (1997)
73008,729,313,3,1998-04-22 23:10:38,Titanic (1997)


In [63]:
def Trend_RE(top_n = 10):
    trd = movie_data[movie_data['timestamp'] > '1998-01-01']
    trd = movie_data.groupby(by = 'title').agg({'rating':['mean','count']})
    trd.columns = ['avg_rating','count']
    trd = trd[trd['count']> 50]
    trd = trd.sort_values(by = 'count',ascending = False)
    #print(trd,'\n +=================')
    return trd.head(top_n).index.tolist()

Trend_RE()

['Star Wars (1977)',
 'Contact (1997)',
 'Fargo (1996)',
 'Return of the Jedi (1983)',
 'Liar Liar (1997)',
 'English Patient, The (1996)',
 'Scream (1996)',
 'Toy Story (1995)',
 'Air Force One (1997)',
 'Independence Day (ID4) (1996)']

### 3. Contend Recommendation

In [70]:
cv = CountVectorizer(stop_words = 'english')
matrix = cv.fit_transform(movie_data['title'].unique())
contend_similarity = cosine_similarity(matrix)

In [71]:
contend_similarity

array([[1.        , 0.        , 0.        , ..., 0.40824829, 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.        , 0.28867513,
        0.        ],
       ...,
       [0.40824829, 0.        , 0.        , ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.28867513, ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ]], shape=(1664, 1664))

In [74]:
movie_data.head()

,userid,itemid,rating,timestamp,title
0,196,242,3,1997-12-04 15:55:49,Kolya (1996)
1,186,302,3,1998-04-04 19:22:22,L.A. Confidential (1997)
2,22,377,1,1997-11-07 07:18:36,Heavyweights (1994)
3,244,51,2,1997-11-27 05:02:03,Legends of the Fall (1994)
4,166,346,1,1998-02-02 05:33:16,Jackie Brown (1997)


In [78]:
idx = movie_data[movie_data['title'] == "Heavyweights (1994)"].index[0]
idx

np.int64(2)

In [75]:
def condent_based_RE(Movie_name):
    if Movie_name not in movie_data['title'].values:
        return "Movie Not Found"

    idx = movie_data[movie_data['title'] == Movie_name].index[0]
  